### Certain genres from past 10 years will be more popular than others and better for our artist profiles

- artist_genre - use current data
    - standard statistics, averages, describe
    - song popularity

In [4]:
import pandas as pd
import numpy as np
import ast
# for syntax analysis


df = pd.read_csv('../data/raw/track_data_final.csv')

# 1. Extract release year and filter for the last 10 years (2016 - 2026)
df['release_year'] = pd.to_datetime(df['album_release_date'], errors='coerce').dt.year

df_last_10_years = df[df['release_year'] >= 2016].copy()

print(f"Original dataset rows: {len(df)}")
print(f"Filtered dataset rows (last 10 years, >= 2016: {len(df_last_10_years)}")
print("-" * 60)

# 1. Helper function to safely parse genre strings like "['rap', 'hip hop']" or handle missing or special values: '[]'
def parse_genres(genre_val):
    if pd.isna(genre_val) or genre_val == 'N/A' or genre_val == '[]':
        return ['unknown']
    if isinstance(genre_val, list):
        return genre_val if genre_val else ['unknown']
    try:
        # Try evaluating string representation of list
        parsed = ast.literal_eval(genre_val)
        if isinstance(parsed, list) and len(parsed) > 0:
            return parsed
    except:
        pass
    # Fallback for simple comma-separated strings if any
    cleaned = [g.strip(" '\"[]") for g in str(genre_val).split(',') if g.strip()]
    return cleaned if cleaned else ['unknown']

# 2. Create a temporary exploded dataframe for genre analysis
df_temp = df_last_10_years.copy()
df_temp['parsed_genres'] = df_temp['artist_genres'].apply(parse_genres)

# Explode rows so each genre gets its own row per track (supports multiple genres per song)
df_exploded = df_temp.explode('parsed_genres')

print(f"Total track-genre rows after exploding: {len(df_exploded)}")
print("-" * 60)

# 3. Standard statistics and averages for Track Popularity per Genre
print("--- Track Popularity Statistics by Genre ---")
genre_popularity_stats = df_exploded.groupby('parsed_genres')['track_popularity'].agg(
    track_count='count',
    mean_popularity='mean',
    median_popularity='median',
    min_popularity='min',
    max_popularity='max',
    std_popularity='std'
).reset_index()

# Sort by mean popularity descending and filter out extremely rare genres if needed
genre_popularity_stats = genre_popularity_stats.sort_values(by='mean_popularity', ascending=False)

# Display top 15 genres by average track popularity
display(genre_popularity_stats.head(15))

print("-" * 60)
print("--- Summary Describe for Track Popularity overall ---")
display(df['track_popularity'].describe())

Original dataset rows: 8778
Filtered dataset rows (last 10 years, >= 2016: 5256
------------------------------------------------------------
Total track-genre rows after exploding: 6579
------------------------------------------------------------
--- Track Popularity Statistics by Genre ---


,parsed_genres,track_count,mean_popularity,median_popularity,min_popularity,max_popularity,std_popularity
244,pop soul,3,90.333333,90.0,86,95,4.509250
128,funk consciente,1,85.000000,85.0,85,85,NaN
309,trap funk,1,85.000000,85.0,85,85,NaN
269,reggaeton chileno,1,84.000000,84.0,84,84,NaN
7,afro tech,1,82.000000,82.0,82,82,NaN
110,electro corridos,5,81.000000,83.0,71,87,6.928203
87,dembow belico,2,80.000000,80.0,77,83,4.242641
118,europop,1,80.000000,80.0,80,80,NaN
284,sierreño,6,79.333333,79.0,70,87,7.284687
172,indonesian indie,1,79.000000,79.0,79,79,NaN


------------------------------------------------------------
--- Summary Describe for Track Popularity overall ---


count    8778.000000
mean       52.232855
std        24.075221
min         0.000000
25%        39.000000
50%        58.000000
75%        71.000000
max       100.000000
Name: track_popularity, dtype: float64

In [3]:

# 5. Most frequent genres (sorted by track_count descending)
sorted_by_frequency = genre_popularity_stats.sort_values(by='track_count', ascending=False)

print("--- Top Genres Sorted by Frequency (Track Count - Last 10 Years) ---")
display(sorted_by_frequency.head(15))

print("-" * 60)

# 6. Most popular genres with a minimum occurrence threshold (at least x tracks)
min_count_threshold = 100 

popular_frequent_genres = genre_popularity_stats[genre_popularity_stats['track_count'] >= min_count_threshold]
popular_frequent_genres = popular_frequent_genres.sort_values(by='mean_popularity', ascending=False)

print(f"--- Top Popular Genres (appearing at least {min_count_threshold} times - Last 10 Years) ---")
display(popular_frequent_genres.head(15))

--- Top Genres Sorted by Frequency (Track Count - Last 10 Years) ---


,parsed_genres,track_count,mean_popularity,median_popularity,min_popularity,max_popularity,std_popularity
319,unknown,3026,52.161269,59.0,0,100,24.242394
241,pop,238,63.752101,66.5,0,92,18.686213
263,rap,164,67.280488,68.5,0,88,13.866386
292,soundtrack,157,34.458599,38.0,0,71,16.749087
290,soft pop,140,52.607143,60.0,0,88,24.946474
82,dark r&b,116,46.491379,52.0,0,79,20.884567
74,country,112,51.008929,48.0,0,86,16.877671
28,art pop,108,54.175926,58.0,19,91,18.548158
192,latin,91,69.989011,73.0,1,91,15.957231
188,k-pop,87,67.045977,69.0,0,99,19.774842


------------------------------------------------------------
--- Top Popular Genres (appearing at least 100 times - Last 10 Years) ---


,parsed_genres,track_count,mean_popularity,median_popularity,min_popularity,max_popularity,std_popularity
263,rap,164,67.280488,68.5,0,88,13.866386
241,pop,238,63.752101,66.5,0,92,18.686213
28,art pop,108,54.175926,58.0,19,91,18.548158
290,soft pop,140,52.607143,60.0,0,88,24.946474
319,unknown,3026,52.161269,59.0,0,100,24.242394
74,country,112,51.008929,48.0,0,86,16.877671
82,dark r&b,116,46.491379,52.0,0,79,20.884567
292,soundtrack,157,34.458599,38.0,0,71,16.749087


In [5]:
display(df_exploded)

,track_id,track_name,track_number,track_popularity,track_duration_ms,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,release_year,parsed_genres
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,158760,False,BUNT.,69.0,293734.0,['stutter house'],2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,1,single,2023.0,stutter house
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,225328,False,Taylor Swift,100.0,145396321.0,[],4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),2021-04-09,26,album,2021.0,unknown
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72,158463,True,Billie Eilish,90.0,118692183.0,[],0JGOiO34nwfUdDrD612dOp,Happier Than Ever,2021-07-30,16,album,2021.0,unknown
6,41zXlQxzTi6cGAjpOXyLYH,idontwannabeyouanymore,2,78,203569,False,Billie Eilish,90.0,118692183.0,[],7fRrTyKvE4Skh93v97gtcU,dont smile at me,2017-12-22,9,single,2017.0,unknown
7,13jRFAGT8qd6aBwtJySlUm,Allein Allein - BENNETT Remix,1,52,145977,False,Alok,76.0,11247155.0,"['brazilian bass', 'electronic', 'slap house',...",1WKoDELzbFRR6UWNGh50LO,Allein Allein (feat. FREY) [BENNETT Remix],2024-09-27,2,single,2024.0,brazilian bass
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8769,4y4spB9m0Q6026KfkAvy9Q,Lonely (with benny blanco),1,66,149297,True,Justin Bieber,91.0,85050158.0,[],3P5WIUJO0Ots1lQx09TOxk,Lonely (with benny blanco),2020-10-16,1,single,2020.0,unknown
8770,6GFrELGVKBXRcrTNkwjSfF,Body And Mind,3,54,186672,False,girl in red,73.0,8771418.0,['bedroom pop'],69p42ovQdwmYPLUsAQpeNn,if i could make it go quiet,2021-04-30,11,album,2021.0,bedroom pop
8773,58VTP6KnZs12PcAj5rMJ4W,the fruits,1,65,209698,False,Paris Paloma,66.0,786890.0,[],48ssTnnFOCIP17sRaFNqu1,the fruits,2022-04-29,1,single,2022.0,unknown
8774,2dRfUOeTVpr4V4PmmqGkpm,Dark World Theme (The Legend of Zelda),4,2,113143,False,Zelda,8.0,3263.0,[],0JJAfVZtn3EvZnXSQrpLSh,The Legend of Zelda,2023-08-19,7,album,2023.0,unknown
